# How to use regioinvent

Note that running this entire notebook will take you around 30 minutes if you choose the highest cutoff option.

To be able to use regioinvent, you will need:
- to install the brightway2 Python library (brightway2 and NOT brightway2.5), easier is to get it through activity-browser: https://github.com/LCA-ActivityBrowser/activity-browser
- a brightway project within which there is an ecoinvent database with either the version 3.10/3.10.1/3.11/3.12 cutoff
- to download the latest version of the trade database stored here: https://doi.org/10.5281/zenodo.11583814

In [ ]:
import pandas as pd
import numpy as np
import sys
# change the path here to wherever you stored the Regioinvent Python package 
# only needed if you've not installed through pip
# sys.path.append('C://Users/11max/PycharmProjects/Regioinvent/src/')
import regioinvent

The initialization of the Regioinvent class requires 3 arguments:
- the name of the brightway2 project where you stored ecoinvent and where regioinvent will be added
- the name you gave your ecoinvent database within your brightway2 project
- the version of the ecoinvent database ('3.10', '3.10.1', '3.11' or '3.12')

In [ ]:
regio = regioinvent.Regioinvent(
    bw_project_name="bw25_ecoinvent3.12",
    ecoinvent_database_name='ecoinvent-3.12-cutoff',
    ecoinvent_version='3.12'
)

First step of regioinvent is to spatialize the original ecoinvent database. This entails two steps:
- Creating a new biosphere database for spatialized elementary flows (e.g., Ammonia, CA-QC)
- Spatializing the elementary flows used within the ecoinvent database, based on the location of the process itself

In [ ]:
regio.spatialize_my_ecoinvent()

In [ ]:
bw2data.Database('biosphere3_spatialized_flows').random().as_dict()

At this stage, regioinvent writes only one database to your brightway2 project:
- "_biosphere3_spatialized_flows_", which contains all the newly created spatialized elementary flows

The spatialized copy of ecoinvent is prepared fully in memory at this step. It is written later, together with the regioinvent database, once both databases have been fully relinked.

Because elementary flows are now spatialized, you will need a specific LCIA method to operate with any flow (spatialized or not). The following function imports such methods. Currently available: "IW v2.2.1", "EF v3.1", "ReCiPe 2016 v1.03 (H)". Can also import all of them in one go. <br>
<b>Note that the following function will fail for brightway2.5.<b>

In [ ]:
regio.import_fully_regionalized_impact_method()

If you want to go further in the regionalization, i.e., to create new national production processes and to rely on trade data to create regionalized consumption markets of ecoinvent, you can run the next function. There are 3 arguments:
- _trade_database_path_ which is the path where you stored the trade database you downloaded from Zenodo: https://doi.org/10.5281/zenodo.11583814
- _target_database_name_ which is the name that the created regioinvent database will take
- _cutoff_ which is the cut-off (between 0 and 1) beyond which countries will be aggreated as RoW. For more info on what cutoff does, see section "Selection of countries for regionalization" of the Methodology.md file.

This function now runs the in-memory relinking and the final Brightway write for both dependent databases. You no longer need to call `write_database()` separately afterwards.

In [ ]:
regio.regionalize_ecoinvent_with_trade(
    trade_database_path='/Users/romain/GitHub/Regioinvent/dev/trade_data.db',
    target_database_name='Regioinvent99 v1.4',
    cutoff=0.99
)

This call writes both final databases to your brightway project: the spatialized ecoinvent copy (`the-name-of-your-ecoinvent-database - regionalized`) and the regioinvent database (`Regioinvent` in the example above). You can then go on brightway2 or AB to perform your LCAs normally with regioinvent.